In [19]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

# 1.Load data

In [2]:
sysID2uniprot_id = pd.read_csv("/data/c/yangyusheng_optimized/resource/Pombase_FTP/release/2025-05-01/uniprot_id_mapping.tsv", sep="\t", header=None, names=["uniprot_id", "Systematic ID", "Gene"])

sysID2uniprot_id.replace(r"^-$", np.nan, inplace=True, regex=True)

sysID2uniprot_id["Gene"] = sysID2uniprot_id["Gene"].fillna(sysID2uniprot_id["Systematic ID"])

PeptideStats = pd.read_csv("/data/c/yangyusheng_optimized/resource/Pombase_FTP/release/2025-05-01/PeptideStats.tsv", sep="\t")

sysID2uniprot_id_and_stats = sysID2uniprot_id.merge(PeptideStats, left_on="Systematic ID", right_on="Systematic_ID", how="left")

pombe_uniprot_ids = sysID2uniprot_id_and_stats["uniprot_id"].unique().tolist()


## homemade domains

In [3]:
homemade_domains = pd.read_csv("/data/c/yangyusheng_optimized/AFdb_based_domain_extraction/results/UP000002485_284812_SCHPO_v4_and_vDulab_annotated_PAE_domain_based_on_pLDDT.tsv", sep="\t")

non_disordered_domains = homemade_domains.query("domain_type != 'disordered'").copy()

## TED domains

https://zenodo.org/records/13908086

1. ted_id - TED domain identifier in the format AF-<UniProtID>-F1-model_v4_TED<domain_number_in_chain> i.e. AF-A0A1V6M2Y0-F1-model_v4_TED03
2. md5_domain - md5 hash of domain sequence
3. consensus_level - medium (2 methods agreement) or high (3 methods agreement)
4. chopping - domain boundaries in the format <start>-<stop> or <start>-<stop>_<start>-<stop> for discontinuous domains
5. nres_domain - number of residues in domain
6. num_segments - number of individual segments in domain. 
7. plddt - average pLDDT for domain (range from 0 to 100)
8. num_helix_strand_turn - number of helix strand turns predicted by STRIDE
9. num_helix - number of helices predicted by STRIDE
10. num_strand - number of strands predicted by STRIDE
11. num_helix_strand - number of helices and strands predicted by STRIDE
12. num_turn - number of turns predicted by STRIDE
13. proteome_id - proteome identifier in the format proteome-tax_id-<taxonID>-<shard>_v4 i.e. proteome-tax_id-67581-0_v4
14. cath_label - CATH superfamily code if predicted, either a C.A.T.H. homologous superfamily or C.A.T. fold assignment. i.e. 3.40.50.300. Otherwise '-'
15. cath_assignment_level - H for homologous superfamily assignment, T for fold level assignment. Otherwise '-'
16. cath_assignment_method - Method used to assign a CATH label, either foldseek or foldclass. Otherwise '-'
17. packing_density - metric used to determine globularity. A domain with packing_density >=10.333 and norm_rg below 0.356 is considered globular
18. norm_rg - normalised radius of gyration. A domain with packing_density >=10.333 AND norm_rg below 0.356 is considered globular. 
19. tax_common_name - Common name for organism
20. tax_scientific_name - Scientific name for organism
21. tax_lineage - Full taxonomic lineage.

In [4]:
column_names =[
    "ted_id",
    "md5_domain",
    "consensus_level",
    "chopping",
    "nres_domain",
    "num_segments",
    "plddt",
    "num_helix_strand_turn",
    "num_helix",
    "num_strand",
    "num_helix_strand",
    "num_turn",
    "proteome_id",
    "cath_label",
    "cath_assignment_level",
    "cath_assignment_method",
    "packing_density",
    "norm_rg",
    "tax_common_name",
    "tax_scientific_name",
    "tax_lineage",
]

TED_domains = pd.read_csv("/data/c/yangyusheng_optimized/AFdb_based_domain_extraction/references/TED/pombe_ted_domains_detailed.tsv", sep="\t", header=None, names=column_names)

simplified_TED_domains = TED_domains[["ted_id", "consensus_level", "chopping", "nres_domain", "num_segments", "plddt", "packing_density", "norm_rg"]].copy()

simplified_TED_domains["uniprot_id"] = simplified_TED_domains["ted_id"].str.extract("AF-(\S+)-F.*")

simplified_TED_domains.to_csv("../references/TED/simplified_pombe_ted_domains.tsv", sep="\t", index=False)

<>:29: SyntaxWarning: invalid escape sequence '\S'
<>:29: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_4011871/3684806114.py:29: SyntaxWarning: invalid escape sequence '\S'
  simplified_TED_domains["uniprot_id"] = simplified_TED_domains["ted_id"].str.extract("AF-(\S+)-F.*")


## ECOD domains based on DPAM

https://zenodo.org/records/8384983

In [5]:
column_names = [
    "Protein",
    "Column2",
    "Range",
    "Source",
    "ID"
]

ECOD_domains = pd.read_csv("/data/c/yangyusheng_optimized/AFdb_based_domain_extraction/references/ECOD/af2_eukaryotes.dpam_v5.domains.txt", sep=" ", header=None, names=column_names)
ECOD_domains["uniprot_id"] = ECOD_domains["Protein"].str.extract("(\S+)_F.*")

pombe_ECOD_domains = ECOD_domains.query("uniprot_id in @pombe_uniprot_ids").copy()
pombe_ECOD_domains.to_csv("../references/ECOD/pombe_ECOD_domains.tsv", sep="\t", index=False)

<>:10: SyntaxWarning: invalid escape sequence '\S'
<>:10: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_4011871/79034804.py:10: SyntaxWarning: invalid escape sequence '\S'
  ECOD_domains["uniprot_id"] = ECOD_domains["Protein"].str.extract("(\S+)_F.*")


## Protein domains from pombase

In [6]:
protein_domains_from_pombase = pd.read_csv("/data/c/yangyusheng_optimized/resource/Pombase_FTP/release/2025-05-01/ProteinFeatures.tsv", sep="\t")
protein_domains_from_pombase["domain"] = protein_domains_from_pombase["seq_start"].astype(str) + "-" + protein_domains_from_pombase["seq_end"].astype(str)

In [7]:
selected_domain_databases = [
    "PFAM",
    "GENE3D",
    "SUPERFAMILY",
    "PANTHER",
    "FUNFAM",
    "SMART",
    "Prosite",
    "CDD",
    "PRINTS"
]

database_domains = {}
for database in selected_domain_databases:
    database_domains[database] = protein_domains_from_pombase.query("database == @database")

# 2.Plot the domains

| Domain db | uniprot_id column | Column of the residue of the domain |
| --- | --- | --- |
| non_disordered_domains | uniprot_id | domain_residues |
| simplified_TED_domains | uniprot_id | chopping |
| pombe_ECOD_domains | uniprot_id | Range |
| PFAM | None (systematic_id) | domain |
| GENE3D | None (systematic_id) | domain |
| SUPERFAMILY | None (systematic_id) | domain |
| PANTHER | None (systematic_id) | domain |
| FUNFAM | None (systematic_id) | domain |
| SMART | None (systematic_id) | domain |
| Prosite | None (systematic_id) | domain |
| CDD | None (systematic_id) | domain |
| PRINTS | None (systematic_id) | domain |

**Exmaples of the "Residue of the domain" column**

- Continueous domains:
    - 1-101
    - 200-245
- Discontinuous domains:
    - 1-101,130-190
    - 1-101,130-190,200-245


In [8]:
def parse_domain_ranges(range_str):
    """Parse domain range string and return list of (start, end) tuples"""
    if pd.isna(range_str) or range_str == '':
        return []
    
    segments = []
    
    # Handle different separators (comma for homemade, underscore for TED)
    if ',' in str(range_str):
        parts = str(range_str).split(',')
    elif '_' in str(range_str):
        parts = str(range_str).split('_')
    else:
        parts = [str(range_str)]
    
    for part in parts:
        if '-' in part:
            try:
                start, end = map(int, part.split('-'))
                segments.append((start, end))
            except ValueError:
                continue
    
    return segments

def plot_protein_domains(genes, pdf_file, max_proteins=None):
    """Plot domains for selected proteins from all three databases"""
    if max_proteins is not None:
        selected_proteins = genes[:max_proteins]
    else:
        selected_proteins = genes
    
    # Define color palettes for each database
    import matplotlib.cm as cm
    db_colors = {
        'Homemade': cm.get_cmap('Set1'),
        'TED': cm.get_cmap('Set2'), 
        'ECOD': cm.get_cmap('Set3'),
        'PFAM': cm.get_cmap('tab10'),
        'GENE3D': cm.get_cmap('tab20'),
        'SUPERFAMILY': cm.get_cmap('tab20b'),
        'PANTHER': cm.get_cmap('tab20c'),
        'FUNFAM': cm.get_cmap('Pastel1'),
        'SMART': cm.get_cmap('Pastel2'),
        'Prosite': cm.get_cmap('Paired'),
        'CDD': cm.get_cmap('Accent'),
        'PRINTS': cm.get_cmap('Dark2'),
    }

    # each fig contains 10 proteins

    num_of_pages = len(selected_proteins) // 10 + (int(len(selected_proteins) % 10 > 0))

    with PdfPages(pdf_file) as pdf:
        for page in range(num_of_pages):
            start_idx = page * 10
            end_idx = min(start_idx + 10, len(selected_proteins))
            selected_proteins_page = selected_proteins[start_idx:end_idx]
        
            # Create subplots
            fig, axes_raw = plt.subplots(len(selected_proteins_page), 1, figsize=(10, 3*len(selected_proteins_page)))
            # Ensure axes is always a list for consistent indexing
            if len(selected_proteins_page) == 1:
                axes_list = [axes_raw]
            else:
                axes_list = list(axes_raw)
            
            for i, gene in enumerate(selected_proteins_page):
                ax = axes_list[i]
                
                # Get protein length
                protein_info = sysID2uniprot_id_and_stats[sysID2uniprot_id_and_stats['Systematic ID'] == gene]
                if len(protein_info) > 0:
                    protein_length = int(protein_info['Residues'].iloc[0])
                    gene_name = protein_info['Gene'].iloc[0] if not pd.isna(protein_info['Gene'].iloc[0]) else ''
                    sys_id = protein_info['Systematic ID'].iloc[0]
                    uniprot_id = protein_info['uniprot_id'].iloc[0]
                else:
                    protein_length = 1000  # Default if not found
                    gene_name = ''
                    sys_id = ''
                    uniprot_id = ''
                
                # Plot domains from each database
                y_positions = {
                    'Homemade': 23,
                    'TED': 21,
                    'ECOD': 19,
                    'PFAM': 17,
                    'GENE3D': 15,
                    'SUPERFAMILY': 13,
                    'PANTHER': 11,
                    'FUNFAM': 9,
                    'SMART': 7,
                    'Prosite': 5,
                    'CDD': 3,
                    'PRINTS': 1
                }
                bar_height = 1.4
                
                for db_name, df, range_col in [
                    ('Homemade', non_disordered_domains, 'domain_residues'),
                    ('TED', simplified_TED_domains, 'chopping'),
                    ('ECOD', pombe_ECOD_domains, 'Range'),
                    ('PFAM', database_domains['PFAM'], 'domain'),
                    ('GENE3D', database_domains['GENE3D'], 'domain'),
                    ('SUPERFAMILY', database_domains['SUPERFAMILY'], 'domain'),
                    ('PANTHER', database_domains['PANTHER'], 'domain'),
                    ('FUNFAM', database_domains['FUNFAM'], 'domain'),
                    ('SMART', database_domains['SMART'], 'domain'),
                    ('Prosite', database_domains['Prosite'], 'domain'),
                    ('CDD', database_domains['CDD'], 'domain'),
                    ('PRINTS', database_domains['PRINTS'], 'domain')
                ]:
                    if db_name in ["Homemade", "TED", "ECOD"]:
                        protein_domains = df[df['uniprot_id'] == uniprot_id]
                    else:
                        protein_domains = df[df['systematic_id'] == gene]
                    
                    segment_count = 0
                    for _, domain_row in protein_domains.iterrows():
                        segments = parse_domain_ranges(domain_row[range_col])
                        # Use different colors for different segments
                        color = db_colors[db_name](segment_count / 10.0)
                        
                        for segment in segments:
                            start, end = segment
                            ax.barh(y_positions[db_name], end - start + 1, 
                                left=start-1, height=bar_height, 
                                color=color, alpha=0.8, 
                                edgecolor='black', linewidth=0.5)
                        
                        segment_count += 1
                
                # Customize the subplot
                ax.set_xlim(0, protein_length)
                ax.set_ylim(-1, 25)
                ax.set_yticks(np.arange(23, 0, -2))
                ax.set_yticklabels(['Homemade', 'TED', 'ECOD', 'PFAM', 'GENE3D', 
                                    'SUPERFAMILY', 'PANTHER', 'FUNFAM', 
                                    'SMART', 'Prosite', 'CDD', 'PRINTS'], 
                                   fontsize=10, fontweight='bold')
                ax.set_xlabel('Protein Position (residues)', fontsize=12, fontweight='bold')
                
                
                # Create title with gene name and systematic ID
                title_parts = [uniprot_id]
                if gene_name:
                    title_parts.append(f"({gene_name})")
                if sys_id:
                    title_parts.append(f"[{sys_id}]")
                title_parts.append(f"Length: {protein_length}")
                
                ax.set_title(' '.join(title_parts), fontweight='bold', fontsize=16)
                ax.grid(True, alpha=0.3, axis='y')
            
            plt.tight_layout(h_pad=3, pad=3)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    print(f"Saved {pdf_file}")

all_genes = sysID2uniprot_id["Systematic ID"].unique().tolist()
print(f"Found {len(all_genes)} genes with domains across all databases")

# plot_protein_domains(all_genes, "../results/domain_comparison/domain_comparison.pdf")


Found 5119 genes with domains across all databases


# 3. Load A data

In [14]:
insertion_fitting_results = pd.read_csv("../../DIT_HAP_pipeline/results/HD_DIT_HAP/16_insertion_level_curve_fitting/insertions_LFC_fitted.tsv", index_col=[0,1,2,3], sep="\t")
insertion_annotations = pd.read_csv("../../DIT_HAP_pipeline/results/HD_DIT_HAP/12_concatenated/annotations.tsv", sep="\t", index_col=[0,1,2,3])
gene_level_fitting_results = pd.read_csv("../../DIT_HAP_pipeline/results/HD_DIT_HAP/18_gene_level_curve_fitting/Gene_level_statistics_fitted.tsv", index_col=0, sep="\t")

in_gene_insertions = insertion_annotations.query("Type != 'Intergenic region' and Distance_to_stop_codon > 4").copy()

in_gene_insertion_fitting_results = insertion_fitting_results.merge(
    in_gene_insertions, 
    left_index=True, 
    right_index=True, 
    how="inner"
)


# 4.Load CKHS regions

In [15]:
CKHS_regions = pd.read_csv("../references/CKHS_region/concated_segments.csv")

In [11]:
def plot_domain_calling_results(gene, resMax, df_est, df_score, ax):

    df_score_1 = df_score[df_score['is.efficient']==1]
    df_score_0 = df_score[df_score['is.efficient']==0]

    est_list = df_est['aa.est'] * (-6)
    score_list_1 = df_score_1['v.filtered']
    aa_list_1 = df_score_1['aa.pos']
    score_list_0 = df_score_0['aver.score']
    aa_list_0 = df_score_0['aa.pos']

    ## Plot segment lines 
    ax.plot(range(0,len(est_list)), est_list, c='r',label='Seg_pattern')

    return ax

# 5.Plot all

In [25]:
def parse_domain_ranges(range_str):
    """Parse domain range string and return list of (start, end) tuples"""
    if pd.isna(range_str) or range_str == '':
        return []
    
    segments = []
    
    # Handle different separators (comma for homemade, underscore for TED)
    if ',' in str(range_str):
        parts = str(range_str).split(',')
    elif '_' in str(range_str):
        parts = str(range_str).split('_')
    else:
        parts = [str(range_str)]
    
    for part in parts:
        if '-' in part:
            try:
                start, end = map(int, part.split('-'))
                segments.append((start, end))
            except ValueError:
                continue
    
    return segments

def plot_protein_domains(genes, pdf_file, feature, max_genes=None):
    """Plot domains for selected proteins from all three databases"""
    if max_genes is not None:
        selected_genes = genes[:max_genes]
    else:
        selected_genes = genes
    
    # Define color palettes for each database
    import matplotlib.cm as cm
    db_colors = {
        'CKHS': cm.get_cmap('Set1'),
        'Homemade': cm.get_cmap('Set2'),
        'TED': cm.get_cmap('Set3'), 
        'ECOD': cm.get_cmap('Paired'),
        'PFAM': cm.get_cmap('tab10'),
        'GENE3D': cm.get_cmap('tab20'),
        'SUPERFAMILY': cm.get_cmap('tab20b'),
        'PANTHER': cm.get_cmap('tab20c'),
        'FUNFAM': cm.get_cmap('Pastel1'),
        'SMART': cm.get_cmap('Pastel2'),
        'Prosite': cm.get_cmap('Accent'),
        'CDD': cm.get_cmap('Dark2'),
        'PRINTS': cm.get_cmap('Accent')
    }

    # each fig contains 10 proteins

    num_of_pages = len(selected_genes) // 10 + (int(len(selected_genes) % 10 > 0))

    with PdfPages(pdf_file) as pdf:
        for page in range(num_of_pages):
            start_idx = page * 10
            end_idx = min(start_idx + 10, len(selected_genes))
            selected_gene_page = selected_genes[start_idx:end_idx]
        
            # Create subplots
            fig, axes_raw = plt.subplots(5, 2, figsize=(10, 15))
            # Ensure axes is always a list for consistent indexing
            if len(selected_gene_page) == 1:
                axes_list = axes_raw.flatten()
            else:
                axes_list = axes_raw.flatten()
            
            for i, gene in enumerate(selected_gene_page):
                ax = axes_list[i]
                
                # Get protein length
                protein_info = sysID2uniprot_id_and_stats[sysID2uniprot_id_and_stats['Gene'] == gene]
                insertion_in_current_gene = in_gene_insertion_fitting_results.query("Name == @gene")

                forward_insertion = insertion_in_current_gene.query("Insertion_direction == 'Forward'").copy()
                reverse_insertion = insertion_in_current_gene.query("Insertion_direction == 'Reverse'").copy()

                forward_x = forward_insertion['Residue_affected']
                forward_y = forward_insertion[feature]

                reverse_x = reverse_insertion['Residue_affected']   
                reverse_y = reverse_insertion[feature]

                ax.scatter(forward_x, forward_y, s=80, marker='o', label='Forward Insertion', color='#6479cc', alpha=0.7)
                ax.scatter(reverse_x, reverse_y, s=80, marker='o', label='Reverse Insertion', color='#7fb775', alpha=0.7)

                if len(protein_info) > 0:
                    protein_length = int(protein_info['Residues'].iloc[0])
                    gene_name = protein_info['Gene'].iloc[0] if not pd.isna(protein_info['Gene'].iloc[0]) else ''
                    sys_id = protein_info['Systematic ID'].iloc[0]
                    uniprot_id = protein_info['uniprot_id'].iloc[0]
                    gene_fitting_metrics = gene_level_fitting_results.loc[sys_id]
                    essentiality = gene_fitting_metrics["DeletionLibrary_essentiality"]
                    A = gene_fitting_metrics['A']
                    um =  gene_fitting_metrics['um']
                    feature_value = gene_fitting_metrics[feature]
                    ax.axhline(feature_value, color='grey', linestyle='-', linewidth=2, label=f'{feature} (Gene-level)', alpha=0.3)
                else:
                    protein_length = 1000  # Default if not found
                    gene_name = ''
                    sys_id = ''
                    uniprot_id = ''

                CKHS_dir = Path("/data/c/yangyusheng_optimized/Others/protiler/DIT_HAP_test/data/processed/HS_calling_output")
                est_file = CKHS_dir/(gene_name+'_Est.csv')
                score_file = CKHS_dir/(gene_name+'_Score.csv')
                if est_file.exists() and score_file.exists():
                    df_est = pd.read_csv(est_file)
                    df_score = pd.read_csv(score_file)
                    resMax = protein_length
                    ax = plot_domain_calling_results(gene_name, resMax, df_est, df_score, ax)
                
                # Plot domains from each database
                y_positions = {
                    "A": {
                        'CKHS': -1.4 ,
                        'Homemade': -2.1,
                        'TED': -2.8,
                        'ECOD': -3.5,
                        'PFAM': -4.2,
                        'GENE3D': -4.9,
                        'SUPERFAMILY': -5.6,
                        'PANTHER': -6.3,
                        'FUNFAM': -7.0,
                        'SMART': -7.7,
                        'Prosite': -8.4,
                        'CDD': -9.1,
                        'PRINTS': -9.8
                    },
                    "um": {
                        'CKHS': -0.5,
                        'Homemade': -0.6,
                        'TED': -0.7,
                        'ECOD': -0.8,
                        'PFAM': -0.9,
                        'GENE3D': -1,
                        'SUPERFAMILY': -1.1,
                        'PANTHER': -1.2,
                        'FUNFAM': -1.3,
                        'SMART': -1.4,
                        'Prosite': -1.5,
                        'CDD': -1.6,
                        'PRINTS': -1.7
                    }
                }
                if feature == "A":
                    bar_height = 0.6
                elif feature == "um":
                    bar_height = 0.08
                
                for db_name, df, range_col in [
                    ('CKHS', CKHS_regions, 'segment'),
                    ('Homemade', non_disordered_domains, 'domain_residues'),
                    ('TED', simplified_TED_domains, 'chopping'),
                    ('ECOD', pombe_ECOD_domains, 'Range'),
                    ('PFAM', database_domains['PFAM'], 'domain'),
                    ('GENE3D', database_domains['GENE3D'], 'domain'),
                    ('SUPERFAMILY', database_domains['SUPERFAMILY'], 'domain'),
                    ('PANTHER', database_domains['PANTHER'], 'domain'),
                    ('FUNFAM', database_domains['FUNFAM'], 'domain'),
                    ('SMART', database_domains['SMART'], 'domain'),
                    ('Prosite', database_domains['Prosite'], 'domain'),
                    ('CDD', database_domains['CDD'], 'domain'),
                    ('PRINTS', database_domains['PRINTS'], 'domain')
                ]:
                    # Get domains for this protein from this database
                    if db_name == 'CKHS':
                        protein_domains = df[df['Gene'] == gene]
                    elif db_name in ["Homemade", "TED", "ECOD"]:
                        protein_domains = df[df['uniprot_id'] == uniprot_id]
                    else:
                        protein_domains = df[df['systematic_id'] == sys_id]
                    
                    domain_count = 0
                    for _, domain_row in protein_domains.iterrows():
                        segments = parse_domain_ranges(domain_row[range_col])
                        
                        for segment in segments:
                            start, end = segment
                            # Use different colors for different domains
                            color = db_colors[db_name](domain_count / 10.0)
                            
                            ax.barh(y_positions[feature][db_name], end - start + 1, 
                                left=start-1, height=bar_height, 
                                color=color, alpha=0.8, 
                                edgecolor='black', linewidth=0.5)
                        
                        domain_count += 1
                
                # Customize the subplot
                max_y = 9 if feature == "A" else 2
                min_y = -11 if feature == "A" else -2
                ax.set_xlim(0, protein_length)
                ax.set_ylim(min_y, max_y)
                ax.axhline(0, color='black', linewidth=1)
                twin_ax = ax.twinx()
                twin_ax.set_ylim(min_y, max_y)
                twin_ax.set_yticks(list(y_positions[feature].values()))
                twin_ax.set_yticklabels(list(y_positions[feature].keys()), fontsize=6, fontweight='bold')
                ax.set_xlabel('Protein Position (residues)', fontsize=8, fontweight='bold')
                
                # Create title with gene name and systematic ID
                title_parts = [uniprot_id]
                if gene_name:
                    title_parts.append(f"({gene_name})")
                if sys_id:
                    title_parts.append(f"[{sys_id}]")
                title_parts.append(f"Length: {protein_length}")
                title_parts.append(f"Essentiality: {essentiality}")
                
                ax.set_title(' '.join(title_parts), fontweight='bold', fontsize=10)
                ax.grid(True, alpha=0.3, axis='y')
            
            plt.tight_layout(h_pad=3, pad=3)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
    print(f"Saved {pdf_file}")

# plot_protein_domains(in_gene_insertion_fitting_results["Name"].unique().tolist(), "../results/domain_comparison/DIT_HAP_with_domain_data3.pdf")


# 6.Plot selected genes

In [ ]:
# selected_genes = [
#     "alm1",
#     "atp25",
#     "bst1",
#     "cdc12",
#     "cdc15",
#     "cdc7",
#     "cdk9",
#     "cid14",
#     "cog6",
#     "cwf24",
#     "cwf3",
#     "dbl2",
#     "dcp1",
#     "dfr1",
#     "esf1",
#     "eso1",
#     "fcp1",
#     "gmh4",
#     "gpi17",
#     "gpi18",
#     "hfi1",
#     "hsf1",
#     "its3",
#     "mcm4",
#     "mdm34",
#     "met9",
#     "mis6",
#     "mms19",
#     "mre11",
#     "nbs1",
#     "not1",
#     "nup189",
#     "pan1",
#     "peg1",
#     "pep3",
#     "rib2",
#     "rib7",
#     "rlp7",
#     "rpt2",
#     "sec8",
#     "sip1",
#     "sog2",
#     "spt7",
#     "ssr2",
#     "tfg1",
#     "tif471",
#     "tpr1",
#     "uba2",
#     "vps11",
#     "vps16",
#     "lsh1",
#     "cdc12",
#     "gpi18",
#     "cid14",
#     "efg1",
#     "tup12",
#     "mrp11",
#     "atp25",
#     "mre11",
#     "sld3",
#     "wrs1",
#     "prh1",
#     "mis6",
#     "meu31",
#     "met9",
#     "cdc15",
#     "taf2",
#     "cdc123",
#     "vps41",
#     "bst1",
#     "rlp7",
#     "bfr2",
#     "ulp2",
#     "swc4",
#     "rpn9",
#     "pst2",
#     "fta2",
#     "cit1",
#     "trl1",
#     "ssr2",
#     "vps11",
#     "rpp101",
#     "vma5",
#     "abo1",
#     "peg1",
#     "ntr1",
#     "alm1",
#     "nup189",
#     "sen1",
#     "rpl702",
#     "mms19",
#     "pub1",
#     "its3",
#     "pan1",
#     "tif471",
#     "tpr1",
#     "hsf1",
#     "sea3",
#     "ino80",
#     "sip1",
#     "mtr10",
#     "uba2",
#     "pdc2",
#     "ucp3",
#     "ste20",
#     "cyr1",
#     "taf51",
#     "cwf24",
#     "naf1",
#     "rsm27",
#     "cog6",
#     "cdc7",
#     "spt7",
#     "imp2",
#     "cbp1",
#     "sog2",
#     "hfi1",
#     "tad2",
#     "ani1",
#     "cnp3",
#     "eso1",
#     "utp6",
#     "cog3",
#     "dbl2",
#     "rib2",
#     "sec8",
#     "asp1",
#     "mcm4",
#     "cwf5",
#     "git3",
#     "ekc1",
#     "ksg1",
#     "tfg1",
#     "bdp1",
#     "pep3",
#     "syp1",
#     "med14",
#     "nse6",
#     "lid2",
#     "cut8",
#     "tfa2",
#     "dia4",
#     "drp1",
#     "mrpl1",
#     "rpc2",
#     "mcb1",
#     "pog1",
#     "mor2",
#     "rpm2",
#     "ssr3",
#     "fab1",
#     "hem15",
#     "pcp1",
#     "ams2",
#     "dbp5",
#     "rpn7",
#     "los1",
#     "zas1",
#     "stt4",
#     "mrpl10",
#     "mrp51",
#     "ero12",
#     "mms1",
#     "raf2",
#     "dna2",
#     "gpi12",
#     "ssr4",
#     "omh6",
#     "cup1",
#     "sin1",
#     "nup120",
#     "tim23",
#     "yta12",
#     "rrp9",
#     "smi1",
#     "rng3",
#     "rad52",
#     "pdi1",
#     "nnk1",
#     "rok1",
#     "alp5",
#     "iml1",
#     "ppp1",
#     "elp1",
#     "urb1",
#     "SPAC8F11.04",
#     "npl4",
#     "cul4",
#     "sgd1",
#     "gta1",
#     "rpc17",
#     "end4",
#     "hmg1",
#     "vam6",
#     "syf2",
#     "isa1",
#     "cwf25",
#     "enp1",
#     "rrp6",
#     "cwf4",
#     "sec39",
#     "sfb3",
#     "pef1",
#     "tti1",
#     "utp14",
#     "mpe1",
#     "gem1",
#     "SPBC15D4.11c",
#     "rna15",
#     "etd1",
#     "eca39",
#     "mrps26",
#     "for3",
#     "fkh2",
#     "cnx1",
#     "rce1",
#     "cut4",
#     "tpz1",
#     "cut20",
#     "lac1",
#     "cut23",
#     "mis18",
#     "mis14",
#     "its8",
#     "srb4",
#     "cnp20",
#     "mrp20",
#     "ned1",
#     "ptr8",
#     "aur1",
#     "vid21",
#     "oct1",
#     "SPAC806.02c",
#     "tim50",
#     "mis19",
#     "tit1",
#     "pep12",
#     "zip1",
#     "myo1",
#     "nak1",
#     "nuc2",
#     "pga1",
#     "smc5",
#     "arp5",
#     "fes1",
#     "ypt1",
#     "nop16",
#     "ebp2",
#     "rgf3",
#     "pkd2",
#     "sfc1",
#     "rps402",
#     "trz1",
#     "vps25",
#     "lcb1",
#     "gea1",
#     "rsm7",
#     "cdc13",
#     "asc1",
#     "ada1",
#     "rng2",
#     "vma8",
#     "hem3",
#     "rad55",
#     "cog2",
#     "ulp1",
#     "msl1",
#     "nup186",
#     "saf3",
#     "pfd6",
#     "cft1",
#     "ppc89",
#     "pob1",
#     "dpm3",
#     "enp2",
#     "nup85",
#     "pir2",
#     "mdj1",
#     "npp106",
#     "fta4",
#     "erd1",
#     "cdc28",
#     "rsd1",
#     "flx1",
#     "SPCC613.08",
#     "plc1",
#     "git7",
#     "pct1",
#     "yml6",
#     "tho7",
#     "tif303",
#     "ini1",
#     "prp43",
#     "grn1",
#     "bpl1",
#     "tho5",
#     "lip2",
#     "rrn7",
#     "scj1",
#     "sec16",
#     "orc4",
#     "ura1",
#     "sap62",
#     "hsp78",
#     "pst1",
#     "cay1",
#     "vps52",
#     "srp21",
#     "amo1",
#     "rrp40",
#     "rps1002",
#     "rct1",
#     "uso1",
#     "prp16",
#     "rsa1",
#     "lip5",
#     "ysh1",
#     "nop52",
#     "prp39",
#     "rqh1",
#     "rcl1",
#     "swf1",
#     "sec27",
#     "nse2",
#     "msw1",
#     "kin1",
#     "cut9",
#     "prp22",
#     "vps51",
#     "pop100",
#     "mob2",
#     "tfb4",
#     "ost3",
#     "spf30",
#     "tho1",
#     "mca1",
#     "cnd3",
#     "emp65",
#     "mcm3",
#     "pfh1",
#     "rfc1",
#     "air1",
#     "uba1",
#     "med18",
#     "pus7",
#     "lam1",
#     "cdc10",
#     "naa20",
#     "brr6",
#     "mis15",
#     "kei1",
#     "taf10",
#     "sec5",
#     "cwf22",
#     "uri1",
#     "shk1",
#     "vma13",
#     "jmj3",
#     "mrpl25",
#     "omh3",
#     "SPBC1861.05",
#     "trz2",
#     "pbn1",
#     "cyb502",
#     "mrpl31",
#     "alp4",
#     "mto1",
#     "sec22",
#     "krs1",
#     "tsc3",
#     "exo8",
#     "mts4",
#     "hrs1",
#     "taf6",
#     "sec3",
#     "ufd1",
#     "pof1",
#     "rpf2",
#     "ync13",
#     "res1",
#     "ckb1",
#     "glo3",
#     "rrp42",
#     "arp8",
#     "itr2",
#     "sam1",
#     "trs23",
#     "vps901",
#     "prp24",
# ]

In [22]:
selected_genes = {
    "Non-essential_N_terminus": ["mis19","rps1002","vps25","tho7","ypt1","rpl702","mrpl1","SPBC15D4.11c","SPCC613.08","pct1","zip1","pbn1","hfi1","fta2","swf1","tad2","rib2","aur1","hem15","eca39","rct1","asc1","dfr1","gpi18","gta1","dia4","tpz1","bdp1","itr2","ulp1","ksg1","ucp3","ulp2","cnp3","shk1","arp8","trz2","dbl2","yta12","ntr1","vps41","eso1","plc1","pep3","vps11","orc4","cdc28","dna2","sen1","pan1"],
    "Non_essential_C_terminus": ["dpm3","rsm27","tsc3","rpp101","msl1","pfd6","mrpl31","ini1","srp21","cyb502","dcp1","trs23","mrpl25","lam1","vps51","mrp20","pga1","naa20","efg1","isa1","cog2","mis18","tho5","rsa1","med18","nop16","mis14","tim23","taf10","sap62","nop52","mrps26","lip2","syf2","ckb1","fta4","kei1","rrp40","mob2","gpi12","rlp7","nse2","mrpl10","rsm7","yml6","cut8","pep12","rib7","rce1","tfa2","fes1","pef1","brr6","tfb4","omh6","rrp42","mrp51","ost3","spf30","air1","rpf2","cup1","cdc123","gmh4","ebp2","hem3","ufd1","rad55","ani1","msw1","rcl1","lip5","erd1","SPAC8F11.04","cwf25","omh3","rpn9","sam1","vma5","ssr4","wrs1","rpn7","scj1","mis15","rna15","ssr3","alp5","tit1","swc4","rpt2","enp1","mdm34","taf6","tim50","sfc1","git3","grn1","cnp20","rok1","mpe1","glo3","pdi1","dbp5","ssr2","smi1","vps52","lcb1","gpi17","naf1","cbp1","nse6","rrp9","mdj1","cwf24","rrn7","tfg1","atp25","srb4","hrs1","cut23","ero12","npl4","krs1","cdk9","met9","rsd1","pof1","ppp1","sec3","SPAC806.02c","hsf1","prp39","nbs1","uba2","enp2","raf2","fkh2","emp65","mre11","nak1","sin1","imp2","cut9","mis6","cwf4","cog6","nup85","pus7","esf1","cid14","ams2","mca1","pop100","sld3","uri1","pkd2","fcp1","its3","rng3","SPBC1861.05","tho1","jmj3","pdc2","oct1","cdc10","sec39","rrp6","trl1","hsp78","ptr8","trz1","sec5","vps16","ekc1","lsh1","cnd3","mcm3","med14","cwf22","sfb3","mts4","vam6","mcm4","tif303","utp14","rfc1","prp24","tpr1","cdc7","mms1","sec8","smc5","end4","bst1","prp22","taf2","abo1","myo1","elp1","rgf3","sea3","tif471","cft1","cut4","peg1","gea1","urb1","nup189","sip1","not1","mor2"],
    "Non_essential_N_terminus_and_C_terminus": ["meu31","vma8","cit1","vps901","taf51","cog3","ppc89","alp4","cwf3","sec27","ada1","pob1","sog2","kin1","spt7","pst2","ste20","pst1","ino80","cdc12"],
    "Non_essential_Center_Region": ["git7","etd1","bfr2","mcb1","mms19","for3","alm1"],
    "Polar_effect": ["cwf5","saf3","rad52","amo1","cdc15","uba1","uso1","nup120"]
}


In [23]:
selected_genes_sort_by_length = sorted(list(set(selected_genes)), key=lambda x: sysID2uniprot_id_and_stats[sysID2uniprot_id_and_stats['Gene'] == x]['Residues'].iloc[0])
print(len(selected_genes_sort_by_length))

IndexError: single positional indexer is out-of-bounds

In [26]:
for type, genes in selected_genes.items():
    plot_protein_domains(genes, f"../results/domain_comparison/DIT_HAP_with_domain_um_{type}.pdf", "um")

/tmp/ipykernel_4011871/1641137440.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_4011871/1641137440.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_4011871/1641137440.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_4011871/1641137440.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

Saved ../results/domain_comparison/DIT_HAP_with_domain_um_Non-essential_N_terminus.pdf


/tmp/ipykernel_4011871/1641137440.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_4011871/1641137440.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_4011871/1641137440.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_4011871/1641137440.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

Saved ../results/domain_comparison/DIT_HAP_with_domain_um_Non_essential_C_terminus.pdf


/tmp/ipykernel_4011871/1641137440.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_4011871/1641137440.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_4011871/1641137440.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_4011871/1641137440.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

Saved ../results/domain_comparison/DIT_HAP_with_domain_um_Non_essential_N_terminus_and_C_terminus.pdf


/tmp/ipykernel_4011871/1641137440.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_4011871/1641137440.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_4011871/1641137440.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_4011871/1641137440.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

Saved ../results/domain_comparison/DIT_HAP_with_domain_um_Non_essential_Center_Region.pdf


/tmp/ipykernel_4011871/1641137440.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_4011871/1641137440.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_4011871/1641137440.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_4011871/1641137440.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

Saved ../results/domain_comparison/DIT_HAP_with_domain_um_Polar_effect.pdf


In [16]:
plot_protein_domains(selected_genes_sort_by_length, "../results/domain_comparison/DIT_HAP_with_domain_A_selected_genes.pdf", "A")

/tmp/ipykernel_3557005/2731916175.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_3557005/2731916175.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_3557005/2731916175.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_3557005/2731916175.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (345 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved ../results/domain_comparison/DIT_HAP_with_domain_A_selected_genes.pdf


In [17]:
plot_protein_domains(selected_genes_sort_by_length, "../results/domain_comparison/DIT_HAP_with_domain_um_selected_genes.pdf", "um")

/tmp/ipykernel_3557005/2731916175.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_3557005/2731916175.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_3557005/2731916175.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_3557005/2731916175.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (345 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved ../results/domain_comparison/DIT_HAP_with_domain_um_selected_genes.pdf


In [21]:
plot_protein_domains(list(set(in_gene_insertion_fitting_results["Name"].unique().tolist()) - set(selected_genes)), "../results/domain_comparison/DIT_HAP_with_domain_data4_without_selected_genes.pdf")

/tmp/ipykernel_3656647/3521346588.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_3656647/3521346588.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_3656647/3521346588.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_3656647/3521346588.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (4,204 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved ../results/domain_comparison/DIT_HAP_with_domain_data4_without_selected_genes.pdf


In [22]:
essential_genes = gene_level_fitting_results.query("A > 3 and um > 0.5")["Name"].tolist()

In [25]:
plot_protein_domains(essential_genes, "../results/domain_comparison/DIT_HAP_with_domain_essential_genes.pdf", "A")

/tmp/ipykernel_464975/2731916175.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'CKHS': cm.get_cmap('Set1'),
/tmp/ipykernel_464975/2731916175.py:37: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'Homemade': cm.get_cmap('Set2'),
/tmp/ipykernel_464975/2731916175.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  'TED': cm.get_cmap('Set3'),
/tmp/ipykernel_464975/2731916175.py:39: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (1,306 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved ../results/domain_comparison/DIT_HAP_with_domain_essential_genes.pdf
